# Pattern 4: Coordinator (Router) Pattern

> **Google Cloud Reference**: [Coordinator Pattern](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#coordinator-pattern)

A **coordinator agent** (powered by an AI model) dynamically analyzes incoming requests and routes them to the right specialized subagent.

**Key distinction from Parallel/Sequential:** The coordinator uses an AI model to make routing decisions — it's not hardcoded logic.

```
User Request
     │
     ▼
[Coordinator Agent] ──── AI model decides routing ────►
         │                                              │
         ├──► [Order Status Agent]                     │
         ├──► [Returns Agent]                          │
         └──► [Refund Agent]                    ───────► Final Response
```

**When to use:** Structured business processes with varied input types requiring adaptive routing.

**Trade-offs:**
- ✅ Flexible — handles diverse inputs without code changes
- ✅ Modular — add new subagents without touching the coordinator
- ⚠️ Higher cost — coordinator + subagent each make model calls
- ⚠️ Routing errors possible if coordinator prompt is weak

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### Example: Customer Service Coordinator
The coordinator determines request type and routes to the appropriate specialist.

In [4]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat

# ── Specialized Subagents ──────────────────────────────────

order_status_agent = Agent(
    name="Order Status Specialist",
    role="Handle order tracking and status inquiries",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    instructions=[
        "Simulate looking up order status in the database.",
        "Orders take 3-5 business days to ship. Standard delivery is 7-10 days.",
        "Always provide estimated delivery date and tracking info.",
        "Apologize for any delays and offer to escalate if order is more than 14 days old.",
    ],
    markdown=True,
)

returns_agent = Agent(
    name="Returns & Exchanges Specialist",
    role="Handle product returns, exchanges, and replacements",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    instructions=[
        "Return window is 30 days from delivery for most items.",
        "Electronics have a 15-day return window.",
        "Items must be in original packaging and condition.",
        "Provide a pre-paid return label for defective items.",
        "Exchanges can be processed immediately — no need to return first.",
    ],
    markdown=True,
)

refund_agent = Agent(
    name="Refunds & Billing Specialist",
    role="Process refunds and resolve billing issues",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    instructions=[
        "Refunds are processed within 5-7 business days.",
        "Credit card refunds appear within 3-5 days after processing.",
        "For billing disputes, escalate amounts over $500 to billing department.",
        "Always confirm the refund amount and payment method before processing.",
        "Proactively offer a store credit option (10% bonus over refund amount).",
    ],
    markdown=True,
)

technical_support_agent = Agent(
    name="Technical Support Specialist",
    role="Handle product technical issues and troubleshooting",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    instructions=[
        "Start with basic troubleshooting: restart, update, reinstall.",
        "If basic steps don't work, escalate to advanced diagnostics.",
        "Provide step-by-step numbered instructions.",
        "Offer a follow-up check-in if the problem might recur.",
    ],
    markdown=True,
)

# ── Coordinator Agent ─────────────────────────────────────
# The coordinator uses an AI model to decide which subagent handles each request.
# This is the key difference from a hardcoded sequential/parallel workflow.

customer_service_team = Team(
    name="Customer Service Coordinator",
    mode="coordinate",  # Coordinator pattern — AI model routes tasks
    model=OpenAIChat(id="google/gemini-2.5-flash"),  # Model for the coordinator
    members=[
        order_status_agent,
        returns_agent,
        refund_agent,
        technical_support_agent,
    ],
    instructions=[
        "Analyze the customer's request and route to the most appropriate specialist.",
        "For order tracking → Order Status Specialist.",
        "For returns/exchanges → Returns & Exchanges Specialist.",
        "For refunds/billing → Refunds & Billing Specialist.",
        "For technical issues → Technical Support Specialist.",
        "If the request spans multiple areas, route to the primary concern first.",
        "Always greet the customer warmly and summarize the resolution at the end.",
    ],
    markdown=True,
)

# Test with different request types — watch the coordinator route them differently
requests = [
    "I ordered a laptop 3 weeks ago and still haven't received it. Order #ORD-98765.",
    "I received a damaged phone. I want to exchange it for a new one.",
    "I was charged twice for my subscription. I need a refund for the duplicate charge.",
    "My smart TV keeps losing WiFi connection after 10 minutes. I've restarted it 5 times.",
]

for i, request in enumerate(requests, 1):
    print(f"\n{'='*65}")
    print(f"REQUEST {i}: {request[:60]}...")
    print('='*65)
    customer_service_team.print_response(request, stream=True)
    print()


REQUEST 1: I ordered a laptop 3 weeks ago and still haven't received it...


Output()



REQUEST 2: I received a damaged phone. I want to exchange it for a new ...


Output()



REQUEST 3: I was charged twice for my subscription. I need a refund for...


Output()



REQUEST 4: My smart TV keeps losing WiFi connection after 10 minutes. I...


Output()

### Coordinator vs Parallel — Key Difference

| Aspect | Coordinator | Parallel |
|--------|------------|----------|
| Routing decision | AI model decides dynamically | Hardcoded — all agents run |
| Which agents run | 1 (the appropriate one) | All members simultaneously |
| Flexibility | High — adapts to new request types | Low — fixed fan-out |
| Cost | Coordinator + 1 subagent | All subagents at once |
| Best for | Diverse requests needing routing | Independent sub-tasks on same input |